## RF-DETR

### Imports

In [1]:
from __future__ import annotations

import json
import shutil
import torch
import colorsys
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict, Counter
import PIL
from PIL import Image, ImageDraw, ImageFont

from rfdetr import RFDETRMedium
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


### Universal Variables

In [2]:
DATASET_DIR = Path('data/RF_DETR/dataset')
RUNS_DIR = Path('runs/RF_DETR')
TRAIN_OUTDIR = RUNS_DIR / 'train'
SPLIT_NAME = 'test'        

RESOLUTION = 640

# for evaluation
WEIGHTS = TRAIN_OUTDIR / 'checkpoint_best_total.pth'
BATCH_SIZE_EVAL = 8
THRESHOLD_EVAL = 0.001     
PRED_JSON = RUNS_DIR / f'{SPLIT_NAME}_predictions_coco.json'

# for artifact generation
ARTIFACTS_DIR = TRAIN_OUTDIR / 'eval'
IOU_FOR_CM = 0.5
VIS_SCORE = 0.25
CURVE_STEPS = 101

IMAGES_DIR = DATASET_DIR / SPLIT_NAME
ANN_JSON = IMAGES_DIR / '_annotations.coco.json'  
OUT_DIR = RUNS_DIR / 'predict'
CONF_THRES = 0.25
MAX_IMAGES = 110  

print('DATASET_DIR:', DATASET_DIR.resolve())
print('TRAIN_OUTDIR:', TRAIN_OUTDIR.resolve())
print('SPLIT_NAME:', SPLIT_NAME)

DATASET_DIR: C:\Users\jerem\Desktop\ARI3129 - Assignment Materials\Submission\data\RF_DETR\dataset
TRAIN_OUTDIR: C:\Users\jerem\Desktop\ARI3129 - Assignment Materials\Submission\runs\RF_DETR\train
SPLIT_NAME: test


### Remapping COCO category IDs to 0...N-1
##### THIS SHOULD ONLY BE RUN ONCE

In [3]:
def remap_coco_ids(coco_path: Path):
    backup = coco_path.with_suffix(coco_path.suffix + ".bak")
    if not backup.exists():
        shutil.copy2(coco_path, backup)
        print(f"Backup created: {backup}")

    data = json.loads(coco_path.read_text(encoding="utf-8"))

    # Collect all category ids present in annotations
    ann_cat_ids = sorted(set(a["category_id"] for a in data.get("annotations", [])))
    if not ann_cat_ids:
        print(f"{coco_path}: no annotations; skipping")
        return

    # Build mapping to 0..K-1
    mapping = {old: new for new, old in enumerate(ann_cat_ids)}
    print(f"{coco_path.name} mapping:", mapping)

    # Remap categories
    if "categories" in data:
        for c in data["categories"]:
            if c["id"] in mapping:
                c["id"] = mapping[c["id"]]

        # Optional: keep categories sorted by id
        data["categories"] = sorted(data["categories"], key=lambda x: x["id"])

    # Remap annotation category_id
    for a in data.get("annotations", []):
        a["category_id"] = mapping[a["category_id"]]

    coco_path.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
    print(f"Rewrote: {coco_path}")

# Setting it to False to prevent accidental execution
DO_REMAP = False
if DO_REMAP:
    root = Path("dataset")
    for split in ['train', 'valid', 'test']: 
        p = root / split / '_annotations.coco.json'
        if p.exists():
            remap_coco_ids(p)


### Try to use GPU

In [3]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print("=== Device check ===")
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version (torch):", torch.version.cuda)
print("Using device:", device)
print()

=== Device check ===
torch.cuda.is_available(): True
GPU: NVIDIA GeForce GTX 1070
CUDA version (torch): 12.1
Using device: cuda:0



### Train RF-DETR

In [ ]:
model = RFDETRMedium(resolution=640)

model.train(
    dataset_dir=DATASET_DIR,                      # path to dataset root containing train/ valid/ test/ folders
    epochs=50,
    batch_size=4,
    grad_accum_steps=4,
    device=device,
    output_dir=str(TRAIN_OUTDIR),
    tensorboard=True,
    early_stopping=True,
    early_stopping_patience=10,
)

Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights
TensorBoard logging initialized. To monitor logs, use 'tensorboard --logdir runs\train' and open http://localhost:6006/ in browser.
Not using distributed mode
git:
  sha: N/A, status: clean, branch: N/A

Namespace(num_classes=3, grad_accum_steps=4, amp=True, lr=0.0001, lr_encoder=0.00015, batch_size=4, weight_decay=0.0001, epochs=50, lr_drop=100, clip_max_norm=0.1, lr_vit_layer_decay=0.8, lr_component_decay=0.7, do_benchmark=False, dropout=0, drop_path=0.0, drop_mode='standard', drop_schedule='constant', cutoff_epoch=0, pretrained_encoder=None, pretrain_weights='rf-detr-medium.pth', pretrain_exclude_keys=None, pretrain_keys_modify_t

Epoch: [0]  [ 0/32]  eta: 0:14:23  lr: 0.000100  class_error: 73.08  loss: 7.6520 (7.6520)  loss_ce: 1.3321 (1.3321)  loss_bbox: 0.0870 (0.0870)  loss_giou: 0.0834 (0.0834)  loss_ce_0: 1.2491 (1.2491)  loss_bbox_0: 0.1199 (0.1199)  loss_giou_0: 0.0983 (0.0983)  loss_ce_1: 1.3436 (1.3436)  loss_bbox_1: 0.0842 (0.0842)  loss_giou_1: 0.0776 (0.0776)  loss_ce_2: 1.3308 (1.3308)  loss_bbox_2: 0.0897 (0.0897)  loss_giou_2: 0.0917 (0.0917)  loss_ce_enc: 1.3302 (1.3302)  loss_bbox_enc: 0.1975 (0.1975)  loss_giou_enc: 0.1369 (0.1369)  loss_ce_unscaled: 1.3321 (1.3321)  class_error_unscaled: 73.0769 (73.0769)  loss_bbox_unscaled: 0.0174 (0.0174)  loss_giou_unscaled: 0.0417 (0.0417)  cardinality_error_unscaled: 3558.5000 (3558.5000)  loss_ce_0_unscaled: 1.2491 (1.2491)  loss_bbox_0_unscaled: 0.0240 (0.0240)  loss_giou_0_unscaled: 0.0492 (0.0492)  cardinality_error_0_unscaled: 3560.0000 (3560.0000)  loss_ce_1_unscaled: 1.3436 (1.3436)  loss_bbox_1_unscaled: 0.0168 (0.0168)  loss_giou_1_unscaled: 0

### Evaluate the trained model on the Test Split

In [4]:
assert ANN_JSON.exists(), f"Annotation file not found: {ANN_JSON}"
assert WEIGHTS.exists(), "No trained model found in runs/train/RFDETRMedium"

model = RFDETRMedium(resolution=RESOLUTION, pretrain_weights=str(WEIGHTS))

coco_gt = COCO(str(ANN_JSON))
img_ids = coco_gt.getImgIds()
imgs = coco_gt.loadImgs(img_ids)

cat_ids = sorted(coco_gt.getCatIds())

results = []

def run_batch(paths, ids):
    dets = model.predict(paths, threshold=THRESHOLD_EVAL)
    if not isinstance(dets, list):
        dets = [dets]

    for image_id, det in zip(ids, dets):
        if det is None or len(det) == 0:
            continue

        for (x1, y1, x2, y2), score, cls in zip(det.xyxy, det.confidence, det.class_id):
            cls = int(cls)
            results.append({
                "image_id": int(image_id),
                "category_id": int(cat_ids[cls]),
                "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],  # xywh
                "score": float(score),
            })

batch_paths, batch_ids = [], []
for im in imgs:
    p = IMAGES_DIR / im["file_name"]
    batch_paths.append(str(p))
    batch_ids.append(im["id"])

    if len(batch_paths) == BATCH_SIZE_EVAL:
        run_batch(batch_paths, batch_ids)
        batch_paths, batch_ids = [], []

if batch_paths:
    run_batch(batch_paths, batch_ids)

PRED_JSON.parent.mkdir(parents=True, exist_ok=True)
PRED_JSON.write_text(json.dumps(results))
print(f"Saved predictions: {PRED_JSON}  (num detections: {len(results)})")

# COCO evaluation
coco_dt = coco_gt.loadRes(str(PRED_JSON)) if len(results) else coco_gt.loadRes([])
coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
coco_eval.params.imgIds = img_ids
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights


Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


loading annotations into memory...
Done (t=0.01s)
creating index...
index created!


Saved predictions: runs\RF_DETR\test_predictions_coco.json  (num detections: 33000)
Loading and preparing results...
DONE (t=0.26s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.45s).
Accumulating evaluation results...
DONE (t=0.17s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.928
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.960
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.940
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = -1.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.168
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.966
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.921
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.951
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDe

### Create a Confusion Matrix (with pngs)

In [5]:
def iou_xywh(a, b) -> float:
    ax1, ay1, aw, ah = a
    bx1, by1, bw, bh = b
    ax2, ay2 = ax1 + aw, ay1 + ah
    bx2, by2 = bx1 + bw, by1 + bh

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, aw) * max(0.0, ah)
    area_b = max(0.0, bw) * max(0.0, bh)
    union = area_a + area_b - inter_area
    return 0.0 if union <= 0 else inter_area / union


def confusion_matrix_from_coco(
    ann_path: Path,
    pred_path: Path,
    out_csv: Path,
    out_png: Path,
    out_png_norm: Path,
    iou_thr: float = 0.5,
    score_thr: float = 0.001,
):
    ann_path = Path(ann_path)
    pred_path = Path(pred_path)
    assert ann_path.exists(), f"Missing: {ann_path}"
    assert pred_path.exists(), f"Missing: {pred_path}"

    coco = json.loads(ann_path.read_text(encoding='utf-8'))
    preds = json.loads(pred_path.read_text(encoding='utf-8'))

    categories = sorted(coco['categories'], key=lambda c: c['id'])
    cat_id_to_name = {c['id']: c['name'] for c in categories}
    class_ids = [c['id'] for c in categories]
    class_names = [cat_id_to_name[i] for i in class_ids]

    gt_by_image = defaultdict(list)
    for a in coco['annotations']:
        gt_by_image[a['image_id']].append({'bbox': a['bbox'], 'cat': a['category_id']})

    pred_by_image = defaultdict(list)
    for p in preds:
        if p.get('score', 0.0) < score_thr:
            continue
        pred_by_image[p['image_id']].append({'bbox': p['bbox'], 'cat': p['category_id'], 'score': p.get('score', 0.0)})

    for img_id in pred_by_image:
        pred_by_image[img_id].sort(key=lambda x: x['score'], reverse=True)

    rows = class_names + ['(bg)']
    cols = class_names + ['(miss)']
    mat = np.zeros((len(rows), len(cols)), dtype=int)

    name_to_row = {n: i for i, n in enumerate(rows)}
    name_to_col = {n: i for i, n in enumerate(cols)}

    for im in coco['images']:
        img_id = im['id']
        gts = gt_by_image.get(img_id, [])
        prs = pred_by_image.get(img_id, [])
        used_pred = set()

        for gt in gts:
            gt_name = cat_id_to_name[gt['cat']]
            best_j, best_iou = None, 0.0
            for j, pr in enumerate(prs):
                if j in used_pred:
                    continue
                i = iou_xywh(gt['bbox'], pr['bbox'])
                if i >= iou_thr and i > best_iou:
                    best_iou, best_j = i, j

            if best_j is None:
                mat[name_to_row[gt_name], name_to_col['(miss)']] += 1
            else:
                used_pred.add(best_j)
                pred_name = cat_id_to_name[prs[best_j]['cat']]
                mat[name_to_row[gt_name], name_to_col[pred_name]] += 1

        for j, pr in enumerate(prs):
            if j not in used_pred:
                pred_name = cat_id_to_name[pr['cat']]
                mat[name_to_row['(bg)'], name_to_col[pred_name]] += 1

    df = pd.DataFrame(mat, index=rows, columns=cols)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_csv)

    def _plot(m, title, path):
        fig = plt.figure(figsize=(10, 7))
        ax = plt.gca()
        im = ax.imshow(m, interpolation='nearest')
        plt.colorbar(im)
        ax.set_xticks(np.arange(len(cols)))
        ax.set_yticks(np.arange(len(rows)))
        ax.set_xticklabels(cols, rotation=90)
        ax.set_yticklabels(rows)
        ax.set_title(title)
        for i in range(m.shape[0]):
            for j in range(m.shape[1]):
                v = m[i, j]
                if v == 0:
                    continue
                ax.text(j, i, f"{v:.2f}" if m.dtype != int else str(int(v)), ha='center', va='center', fontsize=8)
        plt.tight_layout()
        path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(path, dpi=200)
        plt.close(fig)

    _plot(mat, 'Confusion Matrix', out_png)

    m = mat.astype(float)
    denom = m.sum(axis=1, keepdims=True)
    denom[denom == 0] = 1.0
    m_norm = m / denom
    _plot(m_norm, 'Confusion Matrix (Normalized)', out_png_norm)

    return df


# --- RUN CONFUSION MATRIX ---
ANN_PATH = DATASET_DIR / SPLIT_NAME / '_annotations.coco.json'
CM_CSV = RUNS_DIR / 'confusion_matrix.csv'
CM_PNG = RUNS_DIR / 'confusion_matrix.png'
CM_PNG_NORM = RUNS_DIR / 'confusion_matrix_normalized.png'

df_cm = confusion_matrix_from_coco(
    ann_path=ANN_PATH,
    pred_path=PRED_JSON,
    out_csv=CM_CSV,
    out_png=CM_PNG,
    out_png_norm=CM_PNG_NORM,
    iou_thr=IOU_FOR_CM,
    score_thr=THRESHOLD_EVAL,
)

df_cm


,Back,Front,Side,(miss)
Back,33,1,10,0
Front,1,37,4,2
Side,1,0,32,0
(bg),20338,8023,4520,0


### Generate YOLO-style artifacts (jpgs and pngs).

In [6]:
def iou_xywh(a, b) -> float:
    ax1, ay1, aw, ah = a
    bx1, by1, bw, bh = b
    ax2, ay2 = ax1 + aw, ay1 + ah
    bx2, by2 = bx1 + bw, by1 + bh

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0.0, aw) * max(0.0, ah)
    area_b = max(0.0, bw) * max(0.0, bh)
    union = area_a + area_b - inter_area
    return 0.0 if union <= 0 else inter_area / union


def greedy_match_counts(gts, preds, iou_thr: float):
    used = set()
    tp = Counter()
    fp = Counter()
    fn = Counter()

    for gt in gts:
        gt_cat = gt["cat"]
        best_j, best_iou = None, 0.0
        for j, pr in enumerate(preds):
            if j in used:
                continue
            i = iou_xywh(gt["bbox"], pr["bbox"])
            if i >= iou_thr and i > best_iou:
                best_iou, best_j = i, j

        if best_j is None:
            fn[gt_cat] += 1
        else:
            used.add(best_j)
            pr_cat = preds[best_j]["cat"]
            if pr_cat == gt_cat:
                tp[gt_cat] += 1
            else:
                fn[gt_cat] += 1
                fp[pr_cat] += 1

    for j, pr in enumerate(preds):
        if j not in used:
            fp[pr["cat"]] += 1

    return tp, fp, fn


def load_coco(ann_path: Path):
    data = json.loads(ann_path.read_text(encoding="utf-8"))
    cat_id_to_name = {c["id"]: c.get("name", str(c["id"])) for c in data.get("categories", [])}
    img_id_to_file = {im["id"]: im["file_name"] for im in data.get("images", [])}

    gt_by_image = defaultdict(list)
    class_counts = Counter()

    for a in data.get("annotations", []):
        gt_by_image[a["image_id"]].append({"bbox": a["bbox"], "cat": a["category_id"]})
        class_counts[a["category_id"]] += 1

    cat_ids = sorted(cat_id_to_name.keys())
    return data, cat_id_to_name, img_id_to_file, gt_by_image, class_counts, cat_ids


def load_preds(pred_path: Path, score_thr: float):
    preds = json.loads(pred_path.read_text(encoding="utf-8"))
    pred_by_image = defaultdict(list)
    for p in preds:
        if p.get("score", 0.0) < score_thr:
            continue
        pred_by_image[p["image_id"]].append({
            "bbox": p["bbox"],
            "cat": p["category_id"],
            "score": float(p.get("score", 0.0)),
        })
    for img_id in pred_by_image:
        pred_by_image[img_id].sort(key=lambda x: x["score"], reverse=True)
    return pred_by_image


def compute_curves(gt_by_image, pred_by_image_all, iou_thr: float, steps: int = 101):
    thresholds = np.linspace(0.0, 1.0, steps)
    rows = []
    for t in thresholds:
        TP = FP = FN = 0
        for img_id, gts in gt_by_image.items():
            preds = [p for p in pred_by_image_all.get(img_id, []) if p["score"] >= t]
            preds.sort(key=lambda x: x["score"], reverse=True)
            tp, fp, fn = greedy_match_counts(gts, preds, iou_thr)
            TP += sum(tp.values())
            FP += sum(fp.values())
            FN += sum(fn.values())

        prec = TP / (TP + FP) if (TP + FP) else 0.0
        rec  = TP / (TP + FN) if (TP + FN) else 0.0
        f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        rows.append((float(t), float(prec), float(rec), float(f1)))

    return pd.DataFrame(rows, columns=["threshold", "precision", "recall", "f1"])


def plot_curve(x, y, title, out_png: Path, xlabel="confidence threshold", ylabel="value"):
    fig = plt.figure(figsize=(7, 5))
    ax = plt.gca()
    ax.plot(x, y)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=200)
    plt.close(fig)


def plot_pr(df, out_png: Path):
    fig = plt.figure(figsize=(7, 5))
    ax = plt.gca()
    ax.plot(df["recall"], df["precision"])
    ax.set_title("BoxPR curve (IoU-matched)")
    ax.set_xlabel("recall")
    ax.set_ylabel("precision")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=200)
    plt.close(fig)


def plot_label_distribution(class_counts: Counter, id_to_name, out_png: Path):
    if not class_counts:
        return
    ids = list(class_counts.keys())
    counts = [class_counts[i] for i in ids]
    names = [id_to_name.get(i, str(i)) for i in ids]

    fig = plt.figure(figsize=(10, 4))
    ax = plt.gca()
    ax.bar(range(len(ids)), counts)
    ax.set_xticks(range(len(ids)))
    ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_title("Label distribution (instances per class)")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=200)
    plt.close(fig)


def generate_yolo_style_artifacts(
    dataset_dir: Path,
    outdir: Path,
    split_name: str,
    pred_json: Path,
    iou_thr: float = 0.5,
    pred_score_curves: float = 0.001,
    curves_steps: int = 101,
):
    dataset_dir = Path(dataset_dir)
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    train_ann = dataset_dir / "train" / "_annotations.coco.json"
    split_ann = dataset_dir / split_name / "_annotations.coco.json"

    assert train_ann.exists(), f"Missing: {train_ann}"
    assert split_ann.exists(), f"Missing: {split_ann}"
    assert pred_json.exists(), f"Missing: {pred_json}"

    _, id_to_name_train, _, _, train_counts, _ = load_coco(train_ann)
    _, _, _, gt_split, _, _ = load_coco(split_ann)

    plot_label_distribution(train_counts, id_to_name_train, outdir / "labels.jpg")

    pred_all = load_preds(pred_json, score_thr=pred_score_curves)

    # Curves
    df = compute_curves(gt_split, pred_all, iou_thr, steps=curves_steps)
    df.to_csv(outdir / "curve_metrics.csv", index=False)

    plot_curve(df["threshold"], df["precision"], "BoxP curve (IoU-matched)", outdir / "BoxP_curve.png", ylabel="precision")
    plot_curve(df["threshold"], df["recall"],    "BoxR curve (IoU-matched)", outdir / "BoxR_curve.png", ylabel="recall")
    plot_curve(df["threshold"], df["f1"],        "BoxF1 curve (IoU-matched)", outdir / "BoxF1_curve.png", ylabel="f1")
    plot_pr(df, outdir / "BoxPR_curve.png")

    print(f"Saved artifacts to: {outdir}")


# --- RUN ARTIFACT GENERATION ---
generate_yolo_style_artifacts(
    dataset_dir=DATASET_DIR,
    outdir=ARTIFACTS_DIR,
    split_name=SPLIT_NAME,
    pred_json=PRED_JSON,
    iou_thr=IOU_FOR_CM,
    pred_score_curves=THRESHOLD_EVAL,
    curves_steps=CURVE_STEPS,
)

Saved artifacts to: runs\train\eval


In [31]:
# ---------- helpers ----------


def _load_font(px: int) -> ImageFont.FreeTypeFont:
    candidates = [
        "DejaVuSans-Bold.ttf",
        "DejaVuSans.ttf",
        "arial.ttf",
        str(Path(PIL.__file__).resolve().parent / "fonts" / "DejaVuSans-Bold.ttf"),
        str(Path(PIL.__file__).resolve().parent / "fonts" / "DejaVuSans.ttf"),
    ]
    for c in candidates:
        try:
            return ImageFont.truetype(c, px)
        except Exception:
            continue

    # Last resort (will be small and not scalable)
    return ImageFont.load_default()



def draw_coco_boxes(img: Image.Image, preds, id2name, conf=0.25, font_scale=0.04, min_font=18):
    draw = ImageDraw.Draw(img)
    w, h = img.size

    lw = max(3, int(round(0.004 * (w + h))))  # slightly thicker boxes
    font_px = max(min_font, int(round(font_scale * h)))
    font = _load_font(font_px)

    pad = max(3, int(round(font_px * 0.35))) # padding scales with font size

    for p in preds:
        score = float(p.get("score", 0.0))
        if score < conf:
            continue

        x, y, bw, bh = p["bbox"]  # COCO format xywh
        x1, y1, x2, y2 = x, y, x + bw, y + bh
        cls_id = int(p["category_id"])
        label = f"{id2name.get(cls_id, cls_id)} {score:.2f}"

        # box
        draw.rectangle([x1, y1, x2, y2], outline=(0, 255, 255), width=lw)

        # label background + text (YOLO-like)
        tb = draw.textbbox((0, 0), label, font=font)
        tw, th = tb[2] - tb[0], tb[3] - tb[1]

        y_text_top = max(0, y1 - th - 2 * pad)
        draw.rectangle(
            [x1, y_text_top, x1 + tw + 2 * pad, y_text_top + th + 2 * pad],
            fill=(0, 255, 255),
        )
        draw.text((x1 + pad, y_text_top + pad), label, fill=(0, 0, 0), font=font)

    return img

# ---------- load mappings ----------
with ANN_JSON.open("r") as f:
    ann = json.load(f)

id2name = {c["id"]: c["name"] for c in ann["categories"]}
imgid2file = {im["id"]: im["file_name"] for im in ann["images"]}

# ---------- load preds and group by image ----------
with PRED_JSON.open("r") as f:
    preds = json.load(f)

preds_by_img = defaultdict(list)
for p in preds:
    preds_by_img[p["image_id"]].append(p)

# ---------- render and save ----------
OUT_DIR.mkdir(parents=True, exist_ok=True)

# optional: render images with most detections first
items = sorted(preds_by_img.items(), key=lambda kv: len(kv[1]), reverse=True)

for _, (img_id, plist) in enumerate(items[:MAX_IMAGES]):   # <-- fixed enumerate unpacking
    fn = imgid2file.get(img_id)
    if not fn:
        continue

    img_path = IMAGES_DIR / fn
    if not img_path.exists():
        continue

    img = Image.open(img_path).convert("RGB")
    img = draw_coco_boxes(
        img,
        plist,
        id2name,
        conf=CONF_THRES,
        font_scale=0.04,   # increase to 0.05–0.06 if you want even bigger
        min_font=18        # increase if needed
    )

    out_path = OUT_DIR / Path(fn).name
    img.save(out_path, quality=90)


### Summary

In [7]:
PRED_PATH = Path(globals().get("PRED_JSON", "test_predictions_coco.json"))

# Adjust this to what you consider a "detected sign"
SCORE_THRESH = 0.25

det_list = json.loads(PRED_PATH.read_text())

# Optional: restrict to the test set image ids, if your notebook has them
# (this avoids counting predictions for any images outside the evaluation subset)
if "img_ids" in globals():
    allowed = set(img_ids)
    det_list = [d for d in det_list if d.get("image_id") in allowed]

# Filter out low-confidence predictions (this fixes the 300-per-image artifact)
det_filt = [d for d in det_list if float(d.get("score", 0.0)) >= SCORE_THRESH]

# Total test images:
# Prefer img_ids if present (true test set size). Otherwise infer from predictions.
if "img_ids" in globals():
    total_test_images = len(img_ids)
    image_id_list = list(img_ids)
else:
    image_id_list = sorted({d["image_id"] for d in det_list})
    total_test_images = len(image_id_list)

# Detections per image (after filtering)
counts_by_image = Counter(d["image_id"] for d in det_filt)
dets_per_image = [counts_by_image.get(i, 0) for i in image_id_list]

total_traffic_signs_detected = int(sum(dets_per_image))
mean_signs_per_image = float(np.mean(dets_per_image)) if total_test_images else 0.0
max_signs_single_image = int(max(dets_per_image)) if total_test_images else 0

# Category distribution (after filtering)
cat_counter = Counter(d["category_id"] for d in det_filt)

# Try to use mapping from your notebook; if not present, show IDs
name_map = globals().get("id2name", {})  # if you built this from ANN_JSON categories earlier

print(f"Total test images: {total_test_images}")
print(f"Total traffic signs detected (score>=thr): {total_traffic_signs_detected}")
print(f"Mean signs per image (score>=thr): {mean_signs_per_image:.3f}")
print(f"Maximum signs in a single image (score>=thr): {max_signs_single_image}")

print("\nDistribution of detected categories (score>=thr):")
for cid, v in cat_counter.most_common():
    print(f"  {name_map.get(cid, str(cid))}: {v}")


Total test images: 110
Total traffic signs detected (score>=thr): 127
Mean signs per image (score>=thr): 1.155
Maximum signs in a single image (score>=thr): 4

Distribution of detected categories (score>=thr):
  0: 45
  1: 43
  2: 39
